In [ ]:
import os
import requests
from bs4 import BeautifulSoup
import time

In [2]:
SCROLL_FILE = "last_scroll_id.txt"
DOMAIN_FILE = "socrata_domains.txt"

def load_state():
    """Load scroll ID + seen domains if resuming."""
    # Load scroll ID
    if os.path.exists(SCROLL_FILE):
        with open(SCROLL_FILE, "r") as f:
            scroll_id = f.read().strip()
            if scroll_id == "":
                scroll_id = "*"   # fallback
    else:
        scroll_id = "*"

    # Load seen domains set
    seen = set()
    if os.path.exists(DOMAIN_FILE):
        with open(DOMAIN_FILE, "r") as f:
            for line in f:
                seen.add(line.strip())

    return scroll_id, seen


def save_scroll_id(scroll_id):
    """Write latest scroll ID to disk so we can resume."""
    with open(SCROLL_FILE, "w") as f:
        f.write(scroll_id)


def get_all_domains_resume():
    url = "https://api.us.socrata.com/api/catalog" # Seems to be a catalog of all things accessible by the api
    limit = 1000 # Limit 1000 because it is small enough to avoid timeouts. 10000 gets timed out. Optimal would probably be between

    # Load previous state
    scroll_id, seen = load_state()

    print(f"Starting with scroll_id={scroll_id}, {len(seen)} domains already saved.")

    # Open output file in append mode
    with open(DOMAIN_FILE, "a") as f_out:

        while True:
            print(f"Fetching scroll_id={scroll_id} ...")

            params = {"scroll_id": scroll_id, "limit": limit}

            try:
                resp = requests.get(url, params=params, timeout=10)
                resp.raise_for_status()
            except Exception as e:
                print(f"Error: {e}, retrying in 5 seconds...")
                time.sleep(5)
                continue

            data = resp.json()
            results = data.get("results", [])

            if not results:
                print("Deep scroll completed or no more results.")
                break

            # Process results
            for item in results:
                metadata = item.get("metadata", {})
                domain = metadata.get("domain")

                if domain and domain not in seen:
                    f_out.write(domain + "\n")
                    f_out.flush()  # ensure immediate write
                    seen.add(domain)

            # Update scroll ID for next request
            next_scroll = results[-1].get("resource").get("id") # id of previous resource can be used to get next scroll
            if not next_scroll:
                print("Finished scrolling dataset.")
                break

            scroll_id = next_scroll
            save_scroll_id(scroll_id)  # persist checkpoint

            time.sleep(0.2) # Only to avoid timeouts, may not be necessary

    print(f"\nCompleted with {len(seen)} total domains.")
    return seen

# Getting all domains to find cities, as the original study was about the state of urban data across US cities, not just NY
domains = get_all_domains_resume() # 555 domains discovered amongst ~220k things in the catalog, but many are for same entity

Starting with scroll_id=zzan-2jsq, 555 domains already saved.
Fetching scroll_id=zzan-2jsq ...
Deep scroll completed or no more results.

Completed with 555 total domains.


In [ ]:
def filter_lines(input_path: str, output_path: str, include: str | None = None, exclude: str | None = None):
    with open(input_path, "r", encoding="utf-8") as infile, \
         open(output_path, "w", encoding="utf-8") as outfile:
        
        for line in infile:
            # If include phrase is given, skip lines that don't contain it
            if include is not None and include not in line:
                continue
            
            # If exclude phrase is given, skip lines that DO contain it
            if exclude is not None and exclude in line:
                continue
            
            outfile.write(line)


In [ ]:
def filter_lines_start(input_path: str, output_path: str, include_start: str | None = None, exclude_start: str | None = None):
    with open(input_path, "r", encoding="utf-8") as infile, \
         open(output_path, "w", encoding="utf-8") as outfile:
        
        for line in infile:
            # Check include-start condition
            if include_start is not None and not line.startswith(include_start):
                continue

            # Check exclude-start condition
            if exclude_start is not None and line.startswith(exclude_start):
                continue

            outfile.write(line)

In [ ]:
filter_lines_start("socrata_domains.txt", "socrata_domains_data.txt", include_start="data.")

In [ ]:
filter_lines("socrata_domains_data.txt", "socrata_domains_countyless.txt", exclude="county")

In [ ]:
def normalize_url(url: str) -> str:
    if not url.startswith(("http://", "https://")):
        return "https://" + url
    return url

def find_city(input_path: str, output_path: str, phrase: str): # Didn't even get all the cities
    print("Not city:")
    with open(input_path, "r", encoding="utf-8") as infile, \
         open(output_path, "w", encoding="utf-8") as outfile:
        
        for line in infile:
            text = ""
            response = requests.get(normalize_url(line.strip()))
            
            if response.ok:

                soup = BeautifulSoup(response.text, "html.parser")

                # Try meta name="title"
                meta_title = soup.find("meta", attrs={"name": "title"}) # Probably need more checks to properly find cities
                if meta_title is not None and "content" in meta_title.attrs:
                    text = meta_title["content"]

                # Fallback to the <title> tag
                elif soup.title:
                    text = soup.title.text.strip()

            if phrase in text.lower():
                outfile.write(line)
            else:
                print(line)

In [ ]:
find_city("socrata_domains_countyless.txt", "socrata_domains_cities_only.txt", phrase="city")

# Metadata needed

- Schema: "columns" | list of columns 

- Column Types: "dataTypeName" | in the list of columns

- Column Names: "name" or "fieldName" | in the list of columns

- Zipcode: "the_geom"

- Nulls: "non_null" and "null" | https://<domain>/resource/<dataset_id>.json?$select=count(*)&$where=<column_name>%20IS%20NULL

- Category: "category" | for top categories of each city

- Format: "displayType" or "viewType"

- Number of Rows: https://<domain>/resource/<dataset_id>.json?$select=count(*)

- Tags: "tags" | list of tags

- Number of Downloads: "downloadCount"

- Number of Views: "viewCount"

- Age of Dataset: "createdAt"

- Update Frequency: "indexUpdatedAt" or "rowsUpdatedAt"